# Data Mining and visualisation - Assignment II: NLP, Clustering
Pablo Ramos - 201885602

In [ ]:
# Libraries
import re
import csv
import sys
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Data Cleaning

## Browsing the data 

It seems to have only two fields: ID and Sentence. It is evidently the transcript for a theatre play, Coriolanus by William Shakespeare; polluted with random lines that seem to have been pulled from a 30s noir drama.

It looks like the goal the clustering algorithm can be separating the lines according to the two (and possibly more) works of literature from the data.

There are special characters, line breaks, and punctuation, which will need to be removed, so the data can be tokenised and fed in a model.

This windowsy line break is in the lines too \r\n 

Keeping this this in mind the cleaning should be as in the next cell:

In [ ]:
def load_data(filepath: str) -> pd.DataFrame:
    """Load the tab-separated data file into a DataFrame."""
    df = pd.read_csv(
        filepath,
        sep="\t",
        names=["ID", "Sentence"],
        skiprows=1, # skip the header row
        encoding="utf-8",
        engine="python",
        quoting=csv.QUOTE_NONE # This makes Pandas ignore the double quotes in the text
    )
    return df


def clean_sentence(text: str) -> str:
    """Clean a single sentence."""
    # Remove literal \n escape sequences (e.g. \\n\\n in row 11)
    text = text.replace("\\n", " ")
    # Remove real newline/carriage return characters (\r\n Windows line endings)
    text = text.replace("\n", " ").replace("\r", " ")
    # Remove punctuation and special characters, keeping only letters, digits, spaces
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    # Collapse multiple spaces into one and strip leading/trailing whitespace
    text = re.sub(r"\s+", " ", text).strip()
    # Lowercase
    text = text.lower()
    return text


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Apply cleaning to the Sentence column."""
    df = df.copy()
    df["Sentence"] = df["Sentence"].astype(str).apply(clean_sentence)
    # Drop rows where cleaning left an empty sentence
    df = df[df["Sentence"].str.len() > 0].reset_index(drop=True)
    return df


if __name__ == "__main__":
    raw = load_data("data/data_train.txt")
    print(f"Loaded {len(raw)} rows")
    print(raw.head())

    cleaned = clean_data(raw)
    print(f"\nCleaned: {len(cleaned)} rows remaining")
    print(cleaned.head())

    cleaned.to_csv("data/data_train_cleaned.csv", index=False)
    print("\nSaved to data/data_train_cleaned.csv")

## Notes on the data cleaning

**`preprocessing.py`**

There are a few steps to processing the data, and these were defined via functions as:

- Loading as a `pd.Dataframe` using `load_data()`
- Removing unwanted characters and replacing upper case with lower case using `clean_sentence`
- Removing empty lines (which result from removing the line breaks and such) with `clean_data` (calls `clean_sentence`)

Everything is then wrapped with:

```py
# Loading the data
raw = load_data("data/data_train.txt")
print(f"Loaded {len(raw)} rows")
print(raw.head())

# Cleaning artifacts, removing empty lines
cleaned = clean_data(raw)
print(f"\nCleaned: {len(cleaned)} rows remaining")
print(cleaned.head())

# Exporting clean data as a csv
cleaned.to_csv("data/data_train_cleaned.csv", index=False)
print("\nSaved to data/data_train_cleaned.csv")
```

# Clustering

## Running from CML

To comply with the given instructions (everything runs with `python clustering.py data.txt`), the data will have to be imported as such:

```py
filepath = sys.argv[1]
raw = load_data(filepath)
cleaned = clean_data(raw)
vectorised_sentences = cleaned["Sentence"].tolist()
```

`sys.argv[1]` should be the path to the  

## Vectorisation

The simpliest way to turn natural language into something numerical is TF-IDF.

A function `vectorise` is used to create `matrix`, I.E. the TF-IDF mathematical abstraction of each sentence.

```py
from sklearn.feature_extraction.text import TfidfVectorizer

def vectorise(sentences: list) -> tuple:
    """Convert cleaned sentences to a TF-IDF matrix (tuple py variable type)."""
    vectorizer = TfidfVectorizer(
        stop_words="english",  # This will get rid of structural words I.E. Articles, common pronouns, &c.
        max_features=4500,     # Keep only the 4500 most important terms, 4500 was picked via trial and error...
        min_df=2,              # Ignores words appearing in fewer than 2 sentences
    )
    matrix = vectorizer.fit_transform(sentences)
    return matrix, vectorizer
```